# Geodesics in Heat: Computing Distance on 3D Triangle Meshes

Based on the paper:
> **"Geodesics in Heat: A New Approach to Computing Distance Based on Heat Flow"**  
> *Keenan Crane (Caltech), Clarisse Weischedel (University of Göttingen), Max Wardetzky (University of Göttingen)*  
> ACM Transactions on Graphics (TOG) / SIGGRAPH 2013

---

In [2]:
from _bootstrap import setup

setup()

PosixPath('/home/arieltr/Projects/master-thesis/source/ai-agent')

In [3]:
import numpy as np
import open3d as o3d

from matplotlib import cm

from geometry.heat_method import HeatMethodSolver

In [ ]:
mesh = o3d.io.read_triangle_mesh(o3d.data.BunnyMesh().path)

mesh.remove_unreferenced_vertices()
mesh.remove_degenerate_triangles()
mesh.remove_duplicated_triangles()
mesh.remove_duplicated_vertices()

mesh_vertices = np.asarray(mesh.vertices)
mesh_triangles = np.asarray(mesh.triangles)

In [4]:
solver = HeatMethodSolver(mesh_vertices, mesh_triangles)

In [5]:
result = solver.compute_distance([0])

In [6]:
import copy

u = result.heat_flow
phi = result.distance

mesh_heat = copy.deepcopy(mesh)
u_compressed = np.maximum(u, 1e-12) ** 0.15
u_min, u_max = u_compressed.min(), u_compressed.max()
u_norm = 1.0 - (u_compressed - u_min) / (u_max - u_min + 1e-10)
mesh_heat.vertex_colors = o3d.utility.Vector3dVector(cm.coolwarm(u_norm)[:, :3])

mesh_dist = copy.deepcopy(mesh)
phi_min, phi_max = phi.min(), phi.max()
phi_norm = (phi - phi_min) / (phi_max - phi_min + 1e-10)

num_stripes = 20.0
phi_stripes = 0.5 + 0.5 * np.sin(2.0 * np.pi * num_stripes * phi_norm)
mesh_dist.vertex_colors = o3d.utility.Vector3dVector(cm.magma(phi_stripes)[:, :3])

bbox_extent = mesh.get_axis_aligned_bounding_box().get_extent()
offset_x = bbox_extent[0] * 1.2
mesh_dist.translate([offset_x, 0.0, 0.0])

o3d.visualization.draw_geometries([mesh_heat, mesh_dist])